# 02 · Connectome vs controls

Runs an experiment from a config and shows the tables and figures. Same thing as `scripts/run_experiment.py`, just interactive.

If the connectome data isn't downloaded yet it runs the offline synthetic demo instead (the summary says so at the top).

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # configs and data paths are relative to the repo root

from IPython.display import Markdown, display

from flyres import plotting
from flyres.config import load_config
from flyres.connectome import FILES, cache_stem
from flyres.experiment import run_experiment

have_real = Path(f"data/cache/{cache_stem(5)}.npz").exists() or all((Path("data/raw") / f).exists() for f in FILES.values())
CONFIG = "configs/small.yaml" if have_real else "configs/demo_synthetic.yaml"
cfg = load_config(CONFIG)
print("config:", CONFIG)

In [ ]:
result = run_experiment(cfg)  # a few minutes for small.yaml

In [ ]:
display(Markdown(result.summary))

In [ ]:
for ticker in result.ensemble_returns:
    plotting.plot_metric_strip(result.metrics, ticker)

In [ ]:
plotting.plot_memory_curves(result.memory);

In [ ]:
for ticker, ens in result.ensemble_returns.items():
    test = ens.dropna(how="all")
    plotting.plot_equity({c: test[c].to_numpy() for c in test.columns}, test.index,
                         title=f"{ticker}: seed-ensemble strategies, growth of 1 (net of costs)")

## What would count as evidence that the fly wiring is special

- The connectome beats **each** control with the same sign across seeds, not just on average.
- It survives multiple comparisons (five controls × several metrics is a lot of chances to get p < 0.05 by luck).
- The seed-ensemble bootstrap interval excludes zero.
- It shows up in memory capacity too, and at a second subgraph size (`--set subgraph.n_neurons=...`).

The most likely outcome on daily SPY is that nothing beats buy & hold or the linear baseline by a meaningful margin. That's fine: memory capacity is the cleaner test of whether biological wiring is special as a reservoir.